# Práctica del lunes resuelta y explicada

Copia didáctica, revisada el 30 de septiembre de 2026. Ejecuta en orden. Conserva los datos del original, corrige el filtro del historial del simulador, completa los ejercicios y añade comprobaciones. Usa biblioteca estándar y simulación local. La conexión real queda explicada al final como lectura.

## Antes de empezar: cómo usar esta práctica

Un notebook mezcla **celdas de texto**, que explican, con **celdas de código**, que Python ejecuta. Ábrelo en Jupyter o en un editor compatible, selecciona un kernel de Python **3.10 o superior** y ejecuta de arriba hacia abajo. Esta versión no necesita instalar paquetes: usa `json` y `re`, que vienen con Python. El original declara Python 3.11; la anotación `str | None` necesita Python 3.10 o superior.

Si ves `NameError`, comprueba si ejecutaste la celda donde se define esa variable o función. Después de cambiar una definición, vuelve a ejecutar esa celda y las que la usan; reiniciar el kernel y ejecutar todo permite comprobar que ninguna variable quedó de una prueba anterior.

El objetivo concreto es responder preguntas de ventas mediante este recorrido: **pregunta → solicitud de herramienta → función Python → resultado → respuesta final**. El simulador representa la parte que decide. El controlador representa el programa que realmente ejecuta. Las celdas originales que se conservan describen también una conexión real; en esta copia, todas las celdas ejecutables trabajan localmente.



# S3·LUN — El loop de tool use, desde cero

**MMIA 6013 · Semana 3 · Lunes**

Construyes la maquinaria completa de un agente: definición de tools, el loop
decidir→ejecutar→observar, y la traza de cada paso.

Sin API key, un `LLMSimulado` emite las mismas estructuras `tool_use` que un
modelo real: **el loop que escribes es idéntico en ambos casos** — esa es la
gracia del ejercicio. Con clave, el mismo loop corre contra OpenAI con
*function calling* de verdad; y si hay VPN, contra la H200 de la universidad
sin gastar un centavo.

## 1. Preparar datos: leer una venta como diccionario

Una fila de `VENTAS` representa una venta. `fecha`, `producto`, `region` y `monto` son claves; los textos y números a su derecha son valores. La lista contiene siete filas, cuatro de marzo y tres de abril.

```python
primera_venta = VENTAS[0]
fecha = primera_venta["fecha"]
monto = primera_venta["monto"]
```

`VENTAS[0]` toma la primera fila; `primera_venta["monto"]` lee 1200. Una lista usa posiciones; un diccionario usa claves.

La configuración original busca una carpeta `curso/actividades` hacia arriba desde el directorio de ejecución. En esta carpeta de notas esa estructura no existe, por lo que fallaría antes de empezar los ejercicios. La copia resuelta conserva el dataset y utiliza `json` y `re` de la biblioteca estándar. La selección de servicios reales se explica como lectura opcional más adelante.


In [1]:
import json
import re

# Dataset original; versión local sin buscar el repositorio del curso.
VENTAS = [
    {"fecha": "2026-03-02", "producto": "laptop", "region": "sierra", "monto": 1200},
    {"fecha": "2026-03-05", "producto": "monitor", "region": "costa", "monto": 300},
    {"fecha": "2026-03-11", "producto": "laptop", "region": "costa", "monto": 1150},
    {"fecha": "2026-03-15", "producto": "teclado", "region": "sierra", "monto": 45},
    {"fecha": "2026-04-01", "producto": "laptop", "region": "oriente", "monto": 1300},
    {"fecha": "2026-04-03", "producto": "monitor", "region": "sierra", "monto": 320},
    {"fecha": "2026-04-18", "producto": "teclado", "region": "costa", "monto": 50},
]


## Parte 1 — Definir las tools

Cada tool = función Python + esquema (nombre, descripción, parámetros).
**La descripción es el prompt:** el modelo elige la tool leyéndola.

## 2. Las herramientas: funciones que sí hacen el trabajo

Una herramienta se define con una función Python y una descripción de sus argumentos. El nombre `consultar_ventas` no hace nada por sí solo; `consultar_ventas("2026-03")` sí ejecuta la función.

`def` define una función: guarda instrucciones para usarlas después. La sangría señala qué líneas pertenecen a esa función y `return` entrega su resultado a quien la llamó. En `mes: str`, `str` indica que se espera texto; `region: str | None = None` indica texto o ausencia de región, con `None` como valor predeterminado. Estas anotaciones documentan tipos; Python no los comprueba automáticamente.

### `consultar_ventas`: filtrar filas

La comprensión de lista recorre cada `v` de `VENTAS` y la conserva si su fecha empieza con el mes pedido y si cumple el filtro opcional de región. Puedes leerla como:

```python
filas = []
for v in VENTAS:
    coincide_mes = v["fecha"].startswith(mes)
    coincide_region = region is None or v["region"] == region
    if coincide_mes and coincide_region:
        filas.append(v)
```

Si `region` vale `None`, no se restringe la región. Si vale `sierra`, solo se conservan sus ventas. `startswith` compara prefijos de texto; no interpreta un calendario. Por eso el controlador añadido verifica `YYYY-MM` antes de despachar.

`return json.dumps(filas, ensure_ascii=False)` devuelve una **cadena que contiene JSON**, no una lista Python. El historial del simulador recibe esa cadena; `json.loads` la convierte otra vez en datos.

### `estadisticas_ventas`: obtener montos y agregarlos

`montos` toma solo el campo `monto` de las filas del mes. `len(montos)` cuenta ventas, `sum(montos)` suma sus importes y la división calcula el promedio por venta. `round(..., 2)` redondea a dos decimales.

Para marzo: `[1200, 300, 1150, 45]`, total `2695`, cantidad `4`, promedio `673.75`. Para abril: `[1300, 320, 50]`, total `1670`, cantidad `3`, promedio `556.67`. Si la lista está vacía, la función devuelve JSON con un campo `error`.

### `listar_productos`: quitar duplicados y ordenar

`{v["producto"] for v in VENTAS}` construye un conjunto: varias ventas de laptop cuentan como un solo nombre. `sorted(...)` convierte esos nombres en la lista ordenada `laptop`, `monitor`, `teclado`.

### `ESQUEMAS` y `TOOLS` cumplen responsabilidades distintas

`ESQUEMAS` contiene nombre, descripción y forma esperada de los argumentos: información para elegir una herramienta. `TOOLS` relaciona nombres con funciones ejecutables: información para que nuestro programa la ejecute.

```python
funcion = TOOLS["estadisticas_ventas"]
resultado = funcion(mes="2026-03")
```

En el esquema, `required: ["mes"]` declara que el mes debe estar presente; `enum` enumera regiones admitidas. Declararlo no obliga a Python a validarlo automáticamente. En la solución, esa validación ocurre en `validar_entrada`.


In [2]:
def consultar_ventas(mes: str, region: str | None = None) -> str:
    filas = [v for v in VENTAS if v["fecha"].startswith(mes)
             and (region is None or v["region"] == region)]
    return json.dumps(filas, ensure_ascii=False)

def estadisticas_ventas(mes: str) -> str:
    montos = [v["monto"] for v in VENTAS if v["fecha"].startswith(mes)]
    if not montos:
        return json.dumps({"error": f"sin datos para {mes}"})
    return json.dumps({"n": len(montos), "total": sum(montos),
                       "promedio": round(sum(montos) / len(montos), 2)})

def listar_productos() -> str:
    return json.dumps(sorted({v["producto"] for v in VENTAS}))

ESQUEMAS = [
    {
        "name": "consultar_ventas",
        "description": ("Devuelve las filas de ventas de un mes dado, opcionalmente filtradas "
                        "por región (sierra, costa u oriente). Úsala cuando el usuario pida "
                        "el detalle de ventas. No calcula agregados."),
        "input_schema": {"type": "object",
                         "properties": {
                             "mes": {"type": "string", "description": "Mes en formato YYYY-MM"},
                             "region": {"type": "string", "enum": ["sierra", "costa", "oriente"]}},
                         "required": ["mes"]},
    },
    {
        "name": "estadisticas_ventas",
        "description": ("Calcula número de ventas, total y promedio de un mes. Úsala cuando "
                        "pidan cifras agregadas ('cuánto vendimos', 'promedio')."),
        "input_schema": {"type": "object",
                         "properties": {"mes": {"type": "string", "description": "YYYY-MM"}},
                         "required": ["mes"]},
    },
    {
        "name": "listar_productos",
        "description": "Lista los productos distintos que aparecen en las ventas.",
        "input_schema": {"type": "object", "properties": {}},
    },
]

TOOLS = {"consultar_ventas": consultar_ventas,
         "estadisticas_ventas": estadisticas_ventas,
         "listar_productos": listar_productos}

### El mismo contrato, dos formatos de cable

`ESQUEMAS` está escrito en el formato de **Anthropic**: `name`, `description` e
`input_schema`. OpenAI pide exactamente la misma información con otra envoltura:
`{"type": "function", "function": {"name", "description", "parameters"}}`.

No es una anécdota de sintaxis. Es **la razón por la que existe MCP**, que se ve el
miércoles: la herramienta es la misma, la descripción es la misma, el esquema JSON es el
mismo, y cada proveedor la quiere envuelta a su manera. Cada integración nueva multiplica
ese trabajo por el número de proveedores. MCP propone declararla una vez.

Quedarse con esta idea hoy hace que el miércoles no sea una API más que aprender.

## 3. Adaptar un esquema: cambiar la envoltura, conservar la información

`a_formato_openai` recorre los esquemas y coloca cada uno dentro de `{"type": "function", "function": ...}`. El campo `input_schema` pasa a llamarse `parameters` en esa envoltura de Chat Completions. Las funciones de ventas no cambian.

La comprensión de lista devuelve una lista nueva. `e` representa un esquema en cada vuelta y `e.get("input_schema", ...)` proporciona un esquema por defecto si falta esa clave.

Las impresiones terminan en `[:150]`: muestran solo los primeros 150 caracteres, por eso ves una descripción recortada. No se ha truncado el esquema almacenado en `ESQUEMAS_OPENAI`.

Esta adaptación prepara descripciones. No solicita al modelo ni ejecuta herramientas. La explicación del protocolo se contrastó con [OpenAI: function calling](https://developers.openai.com/api/docs/guides/function-calling).


In [3]:
def a_formato_openai(esquemas):
    """Traduce los esquemas al formato de OpenAI. La información no cambia; la caja sí."""
    return [{"type": "function",
             "function": {"name": e["name"],
                          "description": e["description"],
                          "parameters": e.get("input_schema", {"type": "object",
                                                               "properties": {}})}}
            for e in esquemas]


ESQUEMAS_OPENAI = a_formato_openai(ESQUEMAS)

print("Anthropic:", json.dumps(ESQUEMAS[1], ensure_ascii=False)[:150], "…")
print("OpenAI   :", json.dumps(ESQUEMAS_OPENAI[1], ensure_ascii=False)[:150], "…")
print(f"\nmisma información, {len(ESQUEMAS)} herramientas, dos envolturas")

Anthropic: {"name": "estadisticas_ventas", "description": "Calcula número de ventas, total y promedio de un mes. Úsala cuando pidan cifras agregadas ('cuánto ven …
OpenAI   : {"type": "function", "function": {"name": "estadisticas_ventas", "description": "Calcula número de ventas, total y promedio de un mes. Úsala cuando pi …

misma información, 3 herramientas, dos envolturas


## Parte 2 — El "LLM": simulado o real

Interfaz común: recibe la conversación (`messages`) y devuelve o bien
`{"tipo": "tool_use", "name": ..., "input": ...}` o bien
`{"tipo": "texto", "texto": ...}`.

## 4. El simulador: pedir una herramienta antes de responder

`LLMSimulado.responder(messages)` lee la pregunta inicial y revisa las herramientas cuyos resultados ya están en el historial. Tiene dos posibles salidas:

```python
{"tipo": "tool_use", "name": "estadisticas_ventas", "input": {"mes": "2026-03"}}
{"tipo": "texto", "texto": "En ese mes hubo 4 ventas..."}
```

La primera es una **solicitud**, todavía sin ejecutar la función. La segunda es un texto final. La clave `tipo` permite que el controlador sepa qué hacer. Estas estructuras son el contrato interno de la práctica; el adaptador real transforma la respuesta HTTP del proveedor a ese contrato antes de entregársela al controlador.

`class LLMSimulado` define un objeto con una operación llamada `responder`. `llm = LLMSimulado()` crea ese objeto. Al llamar `llm.responder(messages)`, Python pasa el propio objeto como `self` y el historial como `messages`; no tienes que escribir `self` en la llamada.

`messages[0]["content"].lower()` toma el contenido del primer mensaje y lo convierte a minúsculas. `ya_llamadas` recorre el historial y recoge los nombres de las herramientas con resultados. Por eso comienza vacío y luego contiene `estadisticas_ventas`.

Los `if` se evalúan en orden: primero cifras agregadas, luego productos, después detalle. Cada `return` termina esa decisión. Las descripciones de `ESQUEMAS` servirían a un modelo real; este simulador elige por condiciones escritas en Python. Por ejemplo, «total y detalle de marzo» entra en la primera condición y solo pide estadísticas.

Para «¿Cuánto vendimos en marzo?», la primera llamada devuelve `tool_use`. Una vez que el controlador agrega el resultado al historial, la siguiente llamada a `responder` convierte el JSON en datos y redacta con `n`, `total` y `promedio`.

### Corrección necesaria del original

El original busca resultados con rol `tool_result_meta`, mientras el ejercicio pide guardar `assistant_tool_use` y `tool_result`. Si sigues literalmente la consigna, el simulador no reconoce que la función ya se ejecutó y la pide otra vez.

En esta copia se cambia el filtro a `role == "tool_result"`. Además, si el resultado trae `error`, el simulador lo informa en lugar de intentar leer cifras inexistentes. Se conserva el comportamiento por reglas: no es un LLM que aprendió a actuar.

El simulador selecciona marzo cuando la pregunta contiene `marzo`; en caso contrario selecciona abril. Solo reconoce `sierra` como filtro de región en sus reglas. Estas limitaciones permanecen visibles para estudiar la diferencia entre un guion y una comprensión general del lenguaje.


In [4]:
class LLMSimulado:
    """Sigue un guion por palabras clave. Emite las MISMAS estructuras que un
    modelo real con function calling — el loop no nota la diferencia."""

    def responder(self, messages):
        pregunta = messages[0]["content"].lower()
        ya_llamadas = [m["name"] for m in messages if m.get("role") == "tool_result"]

        # Ampliación: consumir un resultado de error sin fingir cifras.
        if messages[-1].get("role") == "tool_result":
            resultado = json.loads(messages[-1]["content"])
            if isinstance(resultado, dict) and "error" in resultado:
                return {"tipo": "texto", "texto": f"No pude resolver la consulta: {resultado['error']}"}

        if "cuánto" in pregunta or "total" in pregunta or "promedio" in pregunta:
            if "estadisticas_ventas" not in ya_llamadas:
                mes = "2026-03" if "marzo" in pregunta else "2026-04"
                return {"tipo": "tool_use", "name": "estadisticas_ventas", "input": {"mes": mes}}
            datos = json.loads(messages[-1]["content"])
            return {"tipo": "texto",
                    "texto": f"En ese mes hubo {datos['n']} ventas por un total de "
                             f"${datos['total']} (promedio ${datos['promedio']})."}

        if "productos" in pregunta:
            if "listar_productos" not in ya_llamadas:
                return {"tipo": "tool_use", "name": "listar_productos", "input": {}}
            return {"tipo": "texto", "texto": f"Los productos son: {messages[-1]['content']}"}

        if "detalle" in pregunta or "ventas de" in pregunta:
            if "consultar_ventas" not in ya_llamadas:
                entrada = {"mes": "2026-03" if "marzo" in pregunta else "2026-04"}
                if "sierra" in pregunta:
                    entrada["region"] = "sierra"
                return {"tipo": "tool_use", "name": "consultar_ventas", "input": entrada}
            return {"tipo": "texto", "texto": f"Detalle encontrado: {messages[-1]['content']}"}

        return {"tipo": "texto", "texto": "¿Puedes reformular? Puedo consultar ventas, estadísticas o productos."}

llm = LLMSimulado()
print("Backend: simulador local por reglas")


Backend: simulador local por reglas


## Parte 3 — EL LOOP (tu turno)

### ✏️ Ejercicio 3.1
Implementa `ejecutar_agente(pregunta, max_pasos=5)`:
1. Inicia `messages = [{"role": "user", "content": pregunta}]` y `traza = []`
2. En cada paso: pide respuesta al `llm`.
   - Si es `texto` → registra en la traza y devuélvelo (fin).
   - Si es `tool_use` → ejecuta `TOOLS[name](**input)`, registra el evento
     en la traza, y añade al historial el tool_use y su tool_result.
3. Si llegas a `max_pasos` sin respuesta final, corta y devuélvelo indicado.
   (¡Todo agente necesita un límite de pasos: es tu seguro anti-loop!)

## 5. El bucle: decidir, ejecutar, observar y volver a decidir

`ejecutar_agente` devuelve un diccionario con `estado`, `respuesta`, `pasos`, `traza` y `messages`. Así puedes distinguir una respuesta final de un corte por límite. La copia añade validación de argumentos y resultados de error para que los ejemplos no fallen de manera confusa.

```mermaid
flowchart TD
    P[Pregunta] --> H[Crear historial y traza]
    H --> M[Pedir una decision al simulador]
    M -->|texto| F[Guardar respuesta y finalizar]
    M -->|tool_use| V[Validar nombre y argumentos]
    V --> T[Ejecutar funcion o producir error]
    T --> O[Guardar solicitud y resultado con el mismo id]
    O --> L[Comprobar pasos disponibles]
    L -->|quedan| M
    L -->|agotados| C[Finalizar con max_pasos]
```

### Preparar y contar

`messages` empieza con la pregunta del usuario. `traza=[]` empieza vacía. `range(1, max_pasos + 1)` produce pasos numerados del 1 al máximo incluido.

Aquí **un paso es una decisión del simulador**, no una llamada de herramienta. Una pregunta de ventas normalmente usa una herramienta pero necesita dos decisiones: pedirla y responder con su resultado.

### Tomar una decisión

`decision = llm.responder(messages)` pide qué hacer. Si `tipo` es `texto`, el controlador registra el texto y usa `return` para terminar la función completa. Si es `tool_use`, extrae el nombre y los argumentos.

`decision.get("input", {})` lee una clave con un valor alternativo: si no existe `input`, devuelve un diccionario vacío. La comprobación de protocolo exige un diccionario, un `tipo` reconocido y, en decisiones de texto, un campo `texto` que sea cadena. Una decisión mal formada termina con `error_protocolo`; no se presenta como respuesta final.

### Ejecutar con `**entrada`

```python
nombre = "estadisticas_ventas"
entrada = {"mes": "2026-03"}
resultado = TOOLS[nombre](**entrada)
```

Los corchetes buscan la función por su nombre. `**entrada` reparte las claves del diccionario como argumentos nombrados. Equivale a `estadisticas_ventas(mes="2026-03")`. **Esta línea de nuestro programa ejecuta la herramienta**; el simulador solo solicitó su uso.

La validación anterior rechaza herramientas desconocidas, argumentos inesperados y meses con formato incorrecto. Si falla, se construye un JSON de error. Si una función lanza una excepción, el `try/except` también la convierte en un resultado de error.

### Registrar para que la siguiente decisión tenga datos

Se agregan dos mensajes:

```python
{"role": "assistant_tool_use", "id": "c1", "name": "estadisticas_ventas", "input": {"mes": "2026-03"}}
{"role": "tool_result", "id": "c1", "name": "estadisticas_ventas", "content": "{...resultado...}"}
```

El identificador común enlaza el resultado con la solicitud. El simulador sin ID recibe uno generado como `c1`, `c2`, etc. Un resultado de error también debe tener su pareja.

`messages` contiene la conversación que recibe el simulador en la próxima vuelta; `traza` contiene eventos para inspeccionar lo ocurrido. Son dos listas con usos diferentes.

### Agotar el contador

Si hay una llamada de herramienta en el último paso permitido, se registra su resultado pero no queda otra decisión para redactar. La salida dice `max_pasos` y `respuesta=None`. Ejecutar una herramienta no implica haber completado la pregunta.


In [5]:
def validar_entrada(nombre, entrada):
    if not isinstance(nombre, str) or nombre not in TOOLS:
        return "Herramienta no registrada"
    if not isinstance(entrada, dict):
        return "input debe ser un diccionario"
    permitidos = {
        "consultar_ventas": {"mes", "region"},
        "estadisticas_ventas": {"mes"},
        "listar_productos": set(),
    }
    if set(entrada) - permitidos[nombre]:
        return "Hay argumentos inesperados"
    if nombre != "listar_productos":
        mes = entrada.get("mes")
        if not isinstance(mes, str) or re.fullmatch(r"[0-9]{4}-(0[1-9]|1[0-2])", mes) is None:
            return "mes debe usar YYYY-MM con mes entre 01 y 12"
    region = entrada.get("region")
    if region is not None and region not in ("sierra", "costa", "oriente"):
        return "region debe ser sierra, costa u oriente"
    return None


def ejecutar_agente(pregunta, max_pasos=5):
    if type(max_pasos) is not int or max_pasos < 1:
        raise ValueError("max_pasos debe ser un entero positivo")
    messages = [{"role": "user", "content": pregunta}]
    traza = []
    for paso in range(1, max_pasos + 1):
        decision = llm.responder(messages)
        if (not isinstance(decision, dict)
                or decision.get("tipo") not in ("texto", "tool_use")
                or (decision.get("tipo") == "texto" and not isinstance(decision.get("texto"), str))):
            traza.append({"paso": paso, "tipo": "parada", "motivo": "error_protocolo"})
            return {"estado": "error_protocolo", "respuesta": None,
                    "pasos": paso, "traza": traza, "messages": messages}
        if decision["tipo"] == "texto":
            texto = decision["texto"]
            traza.append({"paso": paso, "tipo": "texto", "texto": texto})
            messages.append({"role": "assistant", "content": texto})
            return {"estado": "respuesta_final", "respuesta": texto,
                    "pasos": paso, "traza": traza, "messages": messages}

        nombre = decision.get("name", "")
        entrada = decision.get("input", {})
        llamada_id = decision.get("id", f"c{paso}")
        error = validar_entrada(nombre, entrada)
        if error:
            resultado = json.dumps({"error": error}, ensure_ascii=False)
        else:
            try:
                resultado = TOOLS[nombre](**entrada)
            except Exception as exc:
                resultado = json.dumps({"error": type(exc).__name__}, ensure_ascii=False)
        messages.append({"role": "assistant_tool_use", "id": llamada_id,
                         "name": nombre, "input": entrada})
        messages.append({"role": "tool_result", "id": llamada_id,
                         "name": nombre, "content": resultado})
        traza.append({"paso": paso, "tipo": "tool_use", "id": llamada_id,
                      "name": nombre, "input": entrada, "resultado": resultado})

    traza.append({"tipo": "parada", "motivo": "max_pasos"})
    return {"estado": "max_pasos", "respuesta": None,
            "pasos": max_pasos, "traza": traza, "messages": messages}


corrida_marzo = ejecutar_agente("¿Cuánto vendimos en marzo?")
traza = corrida_marzo["traza"]
print(corrida_marzo["respuesta"])


En ese mes hubo 4 ventas por un total de $2695 (promedio $673.75).


## Parte 4 — La traza: tu caja negra de vuelo

Cada corrida deja una lista de eventos. Inspecciónala: es lo que en la
Semana 4 enviaremos a Langfuse.

## 6. Leer la traza de marzo

La celda anterior deja `traza = corrida_marzo["traza"]`. Por eso `for e in traza` ahora puede imprimir eventos; en el notebook original esa variable queda sin definir hasta que completes y ejecutes el bucle.

| Paso | Decisión | Ejecución del controlador | Información nueva |
| --- | --- | --- | --- |
| 1 | Pide `estadisticas_ventas` con marzo. | Llama la función y registra solicitud y resultado. | 4 ventas, 2695 total, 673.75 promedio. |
| 2 | Devuelve texto final. | Lo guarda y finaliza. | La respuesta redactada. |

El historial contiene pregunta, solicitud, resultado y respuesta final. La traza resume las dos decisiones. El simulador nunca sumó las ventas: la función hizo esa suma.

Puedes reconstruir la ejecución de marzo así:

1. `messages` contiene solo la pregunta; `ya_llamadas` está vacío. El simulador devuelve una solicitud para `estadisticas_ventas`.
2. El controlador ejecuta la función. La función toma `[1200, 300, 1150, 45]`, suma `2695` y calcula `2695 / 4 = 673.75`.
3. Se agregan la solicitud `c1` y su resultado JSON. El historial ahora tiene tres mensajes. Se registra un evento en `traza`.
4. La segunda decisión recibe ese historial. `ya_llamadas` ya contiene la herramienta y `json.loads(messages[-1]["content"])` lee el resultado. El simulador construye el texto con esos datos.
5. El controlador agrega el texto al historial y devuelve la corrida. Quedan cuatro mensajes, dos eventos de traza y una sola ejecución de herramienta.

`messages[-1]` significa «último mensaje», porque Python permite contar desde el final con índices negativos. `for e in traza` toma un evento por vuelta y `print(e)` lo muestra; imprime diccionarios Python, no un razonamiento oculto del modelo.


In [6]:
for e in traza:
    print(e)

{'paso': 1, 'tipo': 'tool_use', 'id': 'c1', 'name': 'estadisticas_ventas', 'input': {'mes': '2026-03'}, 'resultado': '{"n": 4, "total": 2695, "promedio": 673.75}'}
{'paso': 2, 'tipo': 'texto', 'texto': 'En ese mes hubo 4 ventas por un total de $2695 (promedio $673.75).'}


## Parte 4 — Leer la traza

### ✏️ Ejercicio 4.1
Corre el agente con estas tres preguntas y contesta en un comentario:

1. «¿Cuánto vendimos en marzo?» — una sola herramienta
2. «¿Qué productos tenemos?» — otra herramienta, sin argumentos
3. «¿Cuántos pasos tarda si la pregunta no se puede contestar con las tres?»

Para cada una, mira la **traza**: cuántos pasos dio, qué herramienta pidió, con qué
argumentos, y en qué paso decidió que ya tenía lo necesario. Lo que hay que notar no es la
respuesta final: es que el LLM **nunca ejecutó nada** —solo pidió— y que cada `tool_use`
recibió exactamente un `tool_result`. Esas dos invariantes son todo el contrato del loop.

## 7. Ejercicio 4.1: qué debes observar en cada pregunta

La celda construye una lista de preguntas y usa `{p: ejecutar_agente(p) for p in preguntas}` para guardar una corrida por pregunta. Cada corrida comienza con un historial nuevo; las herramientas son las mismas.

| Pregunta | Herramienta y entrada | Resultado | Decisiones / herramientas |
| --- | --- | --- | --- |
| ¿Cuánto vendimos en marzo? | `estadisticas_ventas`, mes marzo. | 2695 total, 4 ventas, 673.75 promedio. | 2 / 1 |
| ¿Qué productos vendemos? | `listar_productos`, `{}`. | laptop, monitor, teclado. | 2 / 1 |
| Dame el detalle de ventas de marzo en sierra | `consultar_ventas`, marzo y sierra. | laptop 1200 y teclado 45. | 2 / 1 |
| ¿Cuál es el sentido de la vida? | Ninguna. | Pide reformular dentro de su alcance. | 1 / 0 |

Los 1245 del detalle son la suma de esas dos filas, pero `consultar_ventas` devuelve las filas, no ese agregado. La pregunta de productos describe nombres presentes en ventas; no consulta inventario.

Si cambias marzo por abril, la función de estadísticas devuelve 1670 y promedio 556.67. Si pides comparar ambos meses, el simulador no planifica dos consultas: sus reglas son más simples que esa tarea.


In [7]:
preguntas = [
    "¿Cuánto vendimos en marzo?",
    "¿Qué productos vendemos?",
    "Dame el detalle de ventas de marzo en sierra",
    "¿Cuál es el sentido de la vida?",
]
corridas_ejercicio = {p: ejecutar_agente(p) for p in preguntas}
for pregunta, corrida in corridas_ejercicio.items():
    print("PREGUNTA:", pregunta)
    print("RESPUESTA:", corrida["respuesta"])
    print("DECISIONES:", corrida["pasos"], "TOOLS:", sum(e["tipo"] == "tool_use" for e in corrida["traza"]))
    for evento in corrida["traza"]:
        print(evento)


PREGUNTA: ¿Cuánto vendimos en marzo?
RESPUESTA: En ese mes hubo 4 ventas por un total de $2695 (promedio $673.75).
DECISIONES: 2 TOOLS: 1
{'paso': 1, 'tipo': 'tool_use', 'id': 'c1', 'name': 'estadisticas_ventas', 'input': {'mes': '2026-03'}, 'resultado': '{"n": 4, "total": 2695, "promedio": 673.75}'}
{'paso': 2, 'tipo': 'texto', 'texto': 'En ese mes hubo 4 ventas por un total de $2695 (promedio $673.75).'}
PREGUNTA: ¿Qué productos vendemos?
RESPUESTA: Los productos son: ["laptop", "monitor", "teclado"]
DECISIONES: 2 TOOLS: 1
{'paso': 1, 'tipo': 'tool_use', 'id': 'c1', 'name': 'listar_productos', 'input': {}, 'resultado': '["laptop", "monitor", "teclado"]'}
{'paso': 2, 'tipo': 'texto', 'texto': 'Los productos son: ["laptop", "monitor", "teclado"]'}
PREGUNTA: Dame el detalle de ventas de marzo en sierra
RESPUESTA: Detalle encontrado: [{"fecha": "2026-03-02", "producto": "laptop", "region": "sierra", "monto": 1200}, {"fecha": "2026-03-15", "producto": "teclado", "region": "sierra", "monto

## 8. Reflexión del ejercicio: cuándo detenerse y cómo tratar escrituras

La pregunta fuera del dominio recibe un texto final en la primera decisión. Eso es apropiado para este asistente acotado: informa su alcance en vez de inventar una herramienta.

Antes de agregar una función que modifique datos habría que validar su operación y sus permisos; incorporar una revisión humana donde el efecto lo requiera; registrar el resultado; y evitar duplicar escrituras en reintentos mediante idempotencia. Estas son respuestas al ejercicio: la práctica no implementa una función de escritura.


In [8]:
# Respuestas de estudio, no una entrega de Lab 03.
# a) La pregunta fuera del dominio termina en una decisión, sin herramientas.
#    Es razonable pedir reformular porque las tools disponibles solo tratan ventas.
# b) Antes de una tool que escribe: validar operación y permisos; pedir aprobación
#    cuando lo requiera el efecto; usar idempotencia para reintentos y registrar resultado.


## Parte 5 — La taxonomía de la sala de juntas

Fuera de esta clase vas a encontrar la IA explicada en capas. Una versión que
circula mucho en ámbito ejecutivo, y que te van a preguntar en el trabajo:

| Capa | Qué hace | Herramientas que se citan |
|---|---|---|
| 1 · Machine Learning | predice sobre datos históricos | SageMaker, Vertex AI |
| 2 · IA Generativa | genera contenido a partir de un prompt | ChatGPT, Claude, Gemini |
| 3 · Agentes de IA | ejecuta tareas de varios pasos con herramientas | LangChain, CrewAI |
| 4 · Agentic AI | orquesta procesos de negocio completos | Claude Code, Codex |

Es un buen mapa para una conversación de negocio. La pregunta es qué tan bien
aguanta cuando la contrastas con el código que acabas de escribir.

## 9. Taxonomía: responder sin buscar una línea mágica

La tabla de cuatro capas es una clasificación divulgativa del curso. El bucle acotado de ventas encaja en su capa 3; la capa 4 pretende describir una coordinación de procesos más amplia. No existe una modificación puntual que cambie universalmente de categoría al programa.

Un chatbot que solo devuelve texto encaja en la capa 2 de esa tabla. Bajo la definición operativa usada por esta actividad, no realiza el bucle de selección y ejecución de herramientas. Las categorías ayudan a conversar; para evaluar el código debes revisar decisiones, resultados, límites y responsabilidad de ejecución.


In [9]:
# a) Este asistente acotado encaja en la capa 3 de la taxonomía divulgativa.
# b) No hay una línea que lo transforme por sí sola en "capa 4".
# c) La diferencia propuesta es el alcance del proceso de negocio y su coordinación;
#    no una definición técnica universal ni una garantía de autonomía.
# d) Un chatbot sin tools es capa 2 en ese mapa. Bajo la definición operativa del
#    curso, no ejecuta el bucle de elección y uso de herramientas de esta práctica.


## Cierre

- El agente = LLM que decide + tu código que ejecuta + loop con límite de pasos.
- La descripción de la tool es el prompt que gobierna la elección.
- Toda corrida deja traza — sin traza no hay debugging ni observabilidad.
- Un agente se define por la decisión, no por la escala: sin elección de tool
  en bucle no hay agente, por muchas capas que le ponga el diagrama.

**Mañana:** patrones de razonamiento del agente — ReAct, Plan-and-Execute y
Reflexion — y cuándo usar cada uno.

## Lectura opcional — Cómo estaba conectada la variante real del lunes

La celda 1 original localiza la raíz del repositorio y puede leer su `.env`. La celda 7 define `LLMReal` y llama `hacer_llm()`: intenta un servicio configurado y, si no funciona, intenta el servidor universitario antes de utilizar el simulador. Esta copia local selecciona explícitamente `LLMSimulado`.

`LLMReal.responder` convierte los roles internos a mensajes de Chat Completions: la solicitud pasa a `tool_calls` y el resultado a un mensaje `tool` emparejado con `tool_call_id`. Los argumentos del cable se representan como una cadena JSON que debe analizarse. [Guía oficial de function calling](https://developers.openai.com/api/docs/guides/function-calling).

En `_pedir`, `cuerpo` contiene el modelo, historial y esquemas. `json.dumps(...).encode()` prepara bytes para la solicitud HTTP, `Request` construye la petición y `urlopen` realiza la conexión. Un timeout es una espera máxima de red, no un presupuesto de toda la corrida.

El adaptador original toma solo la primera llamada de `tool_calls`; no procesa un conjunto de solicitudes. Además, si los argumentos no son JSON válido, devuelve un texto de error con `tipo="texto"`. En ese caso terminar por texto no significa haber resuelto la pregunta. La selección de parámetros según presencia de clave es una decisión del código docente y debe comprobarse contra el servicio usado.

Los nombres de modelos y direcciones en el original se conservan como datos del curso; no se verificó su disponibilidad. Se incluye el código como lectura, sin ejecutarlo:

```python
class LLMReal:
    """El mismo `responder(messages)`, contra un servicio que habla el protocolo de OpenAI.

    Traduce el historial interno —`assistant_tool_use` y `tool_result`— al del cable, que
    es distinto y conviene mirarlo de cerca:

      · la petición del modelo va en `message.tool_calls`, con los argumentos **como
        cadena JSON**, no como objeto: hay que parsearlos, y un modelo puede devolver JSON
        inválido ahí. Es el primer sitio donde un agente se rompe en producción.
      · el resultado vuelve en un mensaje de rol `tool` con el `tool_call_id` que emparejó.
        Ese emparejamiento es la invariante (2) del loop, escrita en el protocolo.
    """

    def __init__(self, base_url, modelo, api_key=None, etiqueta="API"):
        self.base_url, self.modelo, self.api_key, self.etiqueta = base_url, modelo, api_key, etiqueta

    def _pedir(self, mensajes):
        cuerpo = {"model": self.modelo, "messages": mensajes,
                  "tools": ESQUEMAS_OPENAI, "tool_choice": "auto"}
        # `max_completion_tokens` en OpenAI; el vLLM de la H200 usa `max_tokens`. Mandar el
        # que no toca devuelve 400 en OpenAI, así que se elige por destino.
        cuerpo["max_completion_tokens" if self.api_key else "max_tokens"] = MAX_SALIDA
        cabeceras = {"Content-Type": "application/json"}
        if self.api_key:
            cabeceras["Authorization"] = f"Bearer {self.api_key}"
        pet = urllib.request.Request(f"{self.base_url}/chat/completions",
                                     data=json.dumps(cuerpo).encode(), headers=cabeceras)
        with urllib.request.urlopen(pet, timeout=120) as r:
            return json.loads(r.read())

    def responder(self, messages):
        cable = [{"role": "user", "content": messages[0]["content"]}]
        for m in messages[1:]:
            if m["role"] == "assistant_tool_use":
                cable.append({"role": "assistant", "content": None, "tool_calls": [
                    {"id": m["id"], "type": "function",
                     "function": {"name": m["name"], "arguments": json.dumps(m["input"])}}]})
            elif m["role"] == "tool_result":
                cable.append({"role": "tool", "tool_call_id": m["id"], "content": m["content"]})

        respuesta = self._pedir(cable)
        uso = respuesta.get("usage", {})
        msg = respuesta["choices"][0]["message"]
        print(f"   [{self.etiqueta}] {uso.get('prompt_tokens')} tok entrada / "
              f"{uso.get('completion_tokens')} salida")
        llamadas = msg.get("tool_calls") or []
        if llamadas:
            fn = llamadas[0]["function"]
            try:
                entrada = json.loads(fn["arguments"] or "{}")
            except json.JSONDecodeError:
                # No se inventa un valor: se devuelve el error como texto y el loop termina.
                return {"tipo": "texto",
                        "texto": f"El modelo pidió {fn['name']} con argumentos que no son "
                                 f"JSON válido: {fn['arguments']!r}"}
            return {"tipo": "tool_use", "name": fn["name"], "input": entrada,
                    "id": llamadas[0]["id"]}
        return {"tipo": "texto", "texto": (msg.get("content") or "").strip()}

def hacer_llm():
    """Tres peldaños, del más capaz al que siempre funciona, y dice cuál eligió."""
    clave = os.environ.get("OPENAI_API_KEY")
    if clave:
        try:
            llm = LLMReal("https://api.openai.com/v1", MODELO_API, clave, "API")
            llm._pedir([{"role": "user", "content": "ping"}])       # una prueba, no una fe
            print(f"LLM: OpenAI · {MODELO_API}")
            return llm
        except urllib.error.HTTPError as e:
            print(f"LLM: OpenAI rechazó la petición ({e.code} · {e.read().decode()[:120]})")
        except urllib.error.URLError as e:
            print(f"LLM: sin red hacia OpenAI ({e.reason})")
    try:
        llm = LLMReal(H200, MODELO_H200, None, "H200")
        llm._pedir([{"role": "user", "content": "ping"}])
        print(f"LLM: H200 de la USFQ · {MODELO_H200} (gratis)")
        return llm
    except Exception as e:
        print(f"LLM: la H200 no responde ({type(e).__name__}) — ¿VPN?")
    print("LLM: simulado (determinista, sin costo). El loop es el mismo.")
    return LLMSimulado()

```


## Comprobaciones de los resultados

Estas condiciones deben cumplirse al ejecutar las celdas en orden. `assert` detiene la ejecución si encuentra una condición falsa. Los casos deliberadamente erróneos se comprueban como errores, no como respuestas válidas.


In [10]:
# Invariantes del bucle y resultados del dataset.
assert corrida_marzo["estado"] == "respuesta_final"
assert corrida_marzo["pasos"] == 2
assert json.loads(corrida_marzo["traza"][0]["resultado"]) == {"n": 4, "total": 2695, "promedio": 673.75}
assert json.loads(estadisticas_ventas("2026-04")) == {"n": 3, "total": 1670, "promedio": 556.67}
assert json.loads(listar_productos()) == ["laptop", "monitor", "teclado"]
assert sum(v["monto"] for v in json.loads(consultar_ventas("2026-03", "sierra"))) == 1245
for corrida in corridas_ejercicio.values():
    solicitudes = [m["id"] for m in corrida["messages"] if m["role"] == "assistant_tool_use"]
    resultados = [m["id"] for m in corrida["messages"] if m["role"] == "tool_result"]
    assert solicitudes == resultados
assert corridas_ejercicio["¿Cuál es el sentido de la vida?"]["pasos"] == 1
limite = ejecutar_agente("¿Cuánto vendimos en marzo?", max_pasos=1)
assert limite["estado"] == "max_pasos" and limite["respuesta"] is None

# Un backend de prueba emite una petición y consume su resultado de error.
class ModeloDePrueba:
    def __init__(self, nombre, entrada):
        self.nombre, self.entrada = nombre, entrada

    def responder(self, messages):
        if len(messages) == 1:
            return {"tipo": "tool_use", "name": self.nombre, "input": self.entrada}
        return {"tipo": "texto", "texto": messages[-1]["content"]}

llm_guardado = llm
errores_comprobados = {}
for nombre, entrada in [("no_existe", {}), ("estadisticas_ventas", {}),
                        ("estadisticas_ventas", {"mes": "2026"}),
                        ("estadisticas_ventas", []),
                        ("estadisticas_ventas", {"mes": "2026-13"}),
                        ("consultar_ventas", {"mes": "2026-03", "region": "sur"}),
                        ("listar_productos", {"mes": "2026-03"}),
                        ("estadisticas_ventas", {"mes": "2026-05"})]:
    llm = ModeloDePrueba(nombre, entrada)
    caso = ejecutar_agente("prueba")
    assert "error" in json.loads(caso["traza"][0]["resultado"])
    assert caso["messages"][1]["id"] == caso["messages"][2]["id"]
    errores_comprobados[str((nombre, entrada))] = caso["traza"][0]["resultado"]
llm = llm_guardado

# Mes válido sin ventas: el simulador informa el error, sin intentar leer n/total.
historial_sin_datos = [
    {"role": "user", "content": "¿Cuánto vendimos en mayo?"},
    {"role": "tool_result", "id": "c1", "name": "estadisticas_ventas",
     "content": estadisticas_ventas("2026-05")},
]
assert "sin datos para 2026-05" in LLMSimulado().responder(historial_sin_datos)["texto"]

# Un backend que no aprende del resultado debe detenerse por el límite.
class ModeloRespuestaFija:
    def __init__(self, decision):
        self.decision = decision

    def responder(self, messages):
        return self.decision

try:
    llm = ModeloRespuestaFija({"tipo": "tool_use", "name": "listar_productos", "input": {}})
    repeticion = ejecutar_agente("prueba de repetición", max_pasos=3)
    assert repeticion["estado"] == "max_pasos" and repeticion["pasos"] == 3
    assert [m["id"] for m in repeticion["messages"] if m["role"] == "tool_result"] == ["c1", "c2", "c3"]
    for decision_invalida in [None, {"tipo": "otro"}, {"tipo": "texto"}]:
        llm = ModeloRespuestaFija(decision_invalida)
        assert ejecutar_agente("prueba de protocolo")["estado"] == "error_protocolo"
    for maximo_invalido in [0, -1, 1.5, True]:
        try:
            ejecutar_agente("prueba de límite", max_pasos=maximo_invalido)
        except ValueError:
            pass
        else:
            raise AssertionError("Se aceptó un límite inválido")
finally:
    llm = llm_guardado

resultados_verificados = {
    "tipo": "simulacion local, no evaluacion de un LLM",
    "marzo": corrida_marzo,
    "ejercicio_4_1": corridas_ejercicio,
    "limite_un_paso": limite,
    "errores_controlados": errores_comprobados,
    "repeticion_hasta_limite": repeticion,
    "protocolo_y_limites_invalidos": "rechazados correctamente",
    "verificacion": "Todas las comprobaciones del lunes pasaron",
}
print(resultados_verificados["verificacion"])


Todas las comprobaciones del lunes pasaron
